# kl-divergence-gaussian-closed-form — worked example 3: Two closed-form KL expressions are numerically equivalent

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kl-divergence-gaussian-closed-form`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The closed-form KL can be written in two equivalent ways. The ARENA textbook uses `−0.5*(1 + 2*logsigma − mu^2 − exp(2*logsigma))`, while an alternative form is `0.5*(mu^2 + exp(2*logsigma) − 1 − 2*logsigma)`. Both encode the same quantity and should agree numerically to machine precision.

## Worked solution

Step 1: Implement both forms separately.

Step 2: Compute `form_a = -0.5 * (1 + 2*logsigma - mu**2 - (2*logsigma).exp())` and `form_b = 0.5 * (mu**2 + (2*logsigma).exp() - 1 - 2*logsigma)`.

Step 3: Verify `torch.allclose(form_a, form_b)` — they should be equal up to floating-point rounding.

Step 4: Print both per-sample KLs to show the agreement.

In [ ]:
import torch as t

t.manual_seed(42)

def kl_form_a(mu, logsigma):
    """ARENA/textbook form: -0.5 * (1 + 2*ls - mu^2 - exp(2*ls))"""
    return -0.5 * (1 + 2 * logsigma - mu.pow(2) - (2 * logsigma).exp())

def kl_form_b(mu, logsigma):
    """Alternative form: 0.5 * (mu^2 + exp(2*ls) - 1 - 2*ls)"""
    return 0.5 * (mu.pow(2) + (2 * logsigma).exp() - 1 - 2 * logsigma)

B, D = 6, 8
mu       = t.randn(B, D)
logsigma = t.randn(B, D) * 0.5

kl_a = kl_form_a(mu, logsigma)
kl_b = kl_form_b(mu, logsigma)

print('form_a per-sample mean:', kl_a.sum(dim=1).mean().item())
print('form_b per-sample mean:', kl_b.sum(dim=1).mean().item())
print('max abs diff:', (kl_a - kl_b).abs().max().item())

assert t.allclose(kl_a, kl_b, atol=1e-5), 'Forms disagree!'
print('Forms are numerically equivalent.')